# Day 01: Foundations of State Estimation & Linear Kalman Filter
**State Estimation and Localization for Self-Driving Cars**

Welcome to Day 01 of the 1-week intensive state estimation curriculum. Today we establish the mathematical bedrock:
1. **Mathematical Foundations**: Where does the Measurement Matrix $\mathbf{H}$ come from? 4 real-world automotive calibration examples.
2. **Error Covariance $\mathbf{P}$**: Derivation from Gauss-Markov assumptions and confidence ellipsoids.
3. **Cross-Disciplinary Notation**: Harmonizing symbols across Control, Robotics/SLAM, Aerospace, and Computer Vision.
4. **Recursive Least Squares (RLS)**: Online streaming parameter estimation with forgetting factors.
5. **The Linear Kalman Filter (KF)**: Optimal Bayesian tracking for 2D constant velocity kinematics with interactive Plotly visualizations.

*Note: In accordance with modern interactive visualization standards, this notebook uses **Plotly** exclusively.*


In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from position_class import (
    BatchLeastSquares,
    ohms_law_example,
    wheel_odometry_calibration_example,
    kinematic_position_velocity_example,
    lidar_plane_fitting_example,
    LinearKalmanFilter,
    create_2d_constant_velocity_tracker,
)

print("position_class initialized successfully!")


## 1. Where Does the Measurement Matrix $\mathbf{H}$ Come From?

In linear state estimation, the measurement equation maps the unobserved state vector $\mathbf{x} \in \mathbb{R}^n$ to observed sensor signals $\mathbf{y} \in \mathbb{R}^m$:

$$\mathbf{y} = \mathbf{H} \mathbf{x} + \mathbf{v}, \quad \mathbf{v} \sim \mathcal{N}(\mathbf{0}, \mathbf{R})$$

$\mathbf{H}$ is derived directly from the physical sensing equations.

### Automotive Engineering Examples:
1. **Current Sensing Calibration (Ohm's Law)**: $V_i = I_i \cdot R \implies \mathbf{y} = [V_i], \mathbf{H} = [I_i], x = R$.
2. **Wheel Odometry Calibration**: $v_{\text{meas}, i} = \omega_i \cdot r_{\text{eff}} \implies \mathbf{y} = [v_i], \mathbf{H} = [\omega_i], x = r_{\text{eff}}$.
3. **Vehicle Initial State from GPS**: $p_k = p_0 + v_0 \cdot t_k \implies \mathbf{y} = [p_k], \mathbf{H} = [[1, t_k]], \mathbf{x} = [p_0, v_0]^T$.
4. **LiDAR Ground Plane Fitting**: $z_i = a x_i + b y_i + c \implies \mathbf{y} = [z_i], \mathbf{H} = [[x_i, y_i, 1]], \mathbf{x} = [a, b, c]^T$.


In [ ]:
# Run the 4 automotive calibration examples
r_est, r_std = ohms_law_example()
wheel_r, wheel_std = wheel_odometry_calibration_example()
kin_x, kin_cov = kinematic_position_velocity_example()
lidar_params, lidar_cov = lidar_plane_fitting_example()

print(f"1. Resistor Estimate: {r_est:.3f} ± {r_std:.3f} Ohms (True: ~5.0 Ohms)")
print(f"2. Wheel Radius: {wheel_r:.4f} ± {wheel_std:.4f} m (True: ~0.33 m)")
print(f"3. Initial Pos: {kin_x[0]:.3f} m, Vel: {kin_x[1]:.3f} m/s (True: 10.0 m, 12.0 m/s)")
print(f"4. LiDAR Ground Plane: a={lidar_params[0]:.4f}, b={lidar_params[1]:.4f}, c={lidar_params[2]:.4f}")


## 2. Where Does Error Covariance $\mathbf{P}$ Come From?

The Weighted Least Squares (WLS) solution minimizes:

$$J(\mathbf{x}) = \frac{1}{2} (\mathbf{y} - \mathbf{H}\mathbf{x})^T \mathbf{R}^{-1} (\mathbf{y} - \mathbf{H}\mathbf{x})$$

Setting $\nabla_{\mathbf{x}} J = \mathbf{0}$ yields:

$$\hat{\mathbf{x}} = (\mathbf{H}^T \mathbf{R}^{-1} \mathbf{H})^{-1} \mathbf{H}^T \mathbf{R}^{-1} \mathbf{y}$$

Substituting $\mathbf{y} = \mathbf{H} \mathbf{x}_{\text{true}} + \mathbf{v}$:

$$\hat{\mathbf{x}} - \mathbf{x}_{\text{true}} = (\mathbf{H}^T \mathbf{R}^{-1} \mathbf{H})^{-1} \mathbf{H}^T \mathbf{R}^{-1} \mathbf{v}$$

The posterior covariance matrix $\mathbf{P}$ is the expectation of the outer product of the estimation error:

$$\mathbf{P} = \mathbb{E}[(\hat{\mathbf{x}} - \mathbf{x}_{\text{true}})(\hat{\mathbf{x}} - \mathbf{x}_{\text{true}})^T] = (\mathbf{H}^T \mathbf{R}^{-1} \mathbf{H})^{-1}$$


In [ ]:
# Visualizing LiDAR Ground Plane Fit with Plotly 3D Scatter
from position_class import fit_lidar_ground_plane

np.random.seed(42)
N = 250
# Forward range: 2m to 30m, Lateral range: -4m to +4m
x_pts = np.random.uniform(2.0, 30.0, N)
y_pts = np.random.uniform(-4.0, 4.0, N)

# True road parameters: uphill pitch +1.15 deg (a=0.02), bank/camber roll -0.57 deg (b=-0.01), sensor height 1.70m (c=-1.70)
a_true, b_true, c_true = 0.02, -0.01, -1.70
sigma_lidar = 0.03  # 3cm ranging noise
z_pts = a_true * x_pts + b_true * y_pts + c_true + np.random.normal(0, sigma_lidar, N)

# Run Batch Least Squares Plane Fitting
plane_fit = fit_lidar_ground_plane(x_pts, y_pts, z_pts, sigma_z=sigma_lidar)

print("=== LiDAR Ground Plane Estimation Results ===")
print(f"Estimated Model: z = {plane_fit['a']:.5f}*x + {plane_fit['b']:.5f}*y + {plane_fit['c']:.4f}")
print(f"Road Pitch: {plane_fit['pitch_deg']:.3f}° (True: {np.degrees(np.arctan(a_true)):.3f}°)")
print(f"Road Roll:  {plane_fit['roll_deg']:.3f}° (True: {np.degrees(np.arctan(-b_true)):.3f}°)")
print(f"Sensor Height: {plane_fit['sensor_height_m']:.3f} m (True: 1.700 m)")
print(f"Fit RMSE: {plane_fit['rmse_m']*100:.2f} cm")
print(f"Plane Normal: {plane_fit['normal_vector']}")

# Generate Plane Mesh
xx, yy = np.meshgrid(np.linspace(2.0, 30.0, 15), np.linspace(-4.0, 4.0, 10))
zz = plane_fit['a'] * xx + plane_fit['b'] * yy + plane_fit['c']

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=x_pts, y=y_pts, z=z_pts,
    mode='markers',
    marker=dict(size=3, color=z_pts, colorscale='Turbo', opacity=0.8),
    name='LiDAR Ground Points'
))
fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    opacity=0.6,
    colorscale='Viridis',
    showscale=False,
    name='Estimated Road Plane'
))
fig.update_layout(
    title='<b>LiDAR Ground Plane Least Squares Estimation (ISO 8855 Frame)</b>',
    scene=dict(
        xaxis_title='X [m] (Forward / Longitudinal)',
        yaxis_title='Y [m] (Lateral / Left)',
        zaxis_title='Z [m] (Elevation / Up)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()


## 3. 2D Linear Kalman Filter Tracking

We now test a 4D State Constant Velocity Kalman Filter:

$$\mathbf{x}_k = \begin{bmatrix} p_x \\ p_y \\ v_x \\ v_y \end{bmatrix}_k, \quad
\mathbf{F} = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \quad
\mathbf{H} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$


In [ ]:
# Simulate Vehicle 2D Tracking
np.random.seed(42)
dt = 0.1
total_time = 20.0
steps = int(total_time / dt)

# Ground truth constant velocity with slight sinusoidal perturbation
t = np.linspace(0, total_time, steps)
gt_x = 5.0 * t
gt_y = 20.0 * np.sin(0.2 * t)
gt_vx = np.gradient(gt_x, dt)
gt_vy = np.gradient(gt_y, dt)

# Noisy GPS measurements
sigma_pos = 1.5
meas_x = gt_x + np.random.normal(0, sigma_pos, steps)
meas_y = gt_y + np.random.normal(0, sigma_pos, steps)

# Instantiate Linear Kalman Filter
kf = create_2d_constant_velocity_tracker(dt=dt, sigma_pos_gps=sigma_pos, sigma_acc_process=0.5)
kf.initialize(np.array([meas_x[0], meas_y[0], 0.0, 0.0]), np.eye(4) * 10.0)

est_x, est_y, est_vx, est_vy = [], [], [], []
cov_x, cov_y = [], []

for k in range(steps):
    kf.predict()
    state = kf.update(np.array([meas_x[k], meas_y[k]]))
    est_x.append(state.mean[0])
    est_y.append(state.mean[1])
    est_vx.append(state.mean[2])
    est_vy.append(state.mean[3])
    cov_x.append(np.sqrt(state.covariance[0, 0]))
    cov_y.append(np.sqrt(state.covariance[1, 1]))

# Plot tracking results
fig = make_subplots(rows=2, cols=1, subplot_titles=("<b>2D Vehicle Trajectory Tracking</b>", "<b>Estimated vs Ground Truth Velocities</b>"))

# Subplot 1: Trajectory
fig.add_trace(go.Scatter(x=gt_x, y=gt_y, mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=meas_x, y=meas_y, mode='markers', name='Noisy GPS Measurements', marker=dict(color='red', size=4, opacity=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=est_x, y=est_y, mode='lines', name='Kalman Filter Estimate', line=dict(color='blue', width=2)), row=1, col=1)

# Subplot 2: Velocities
fig.add_trace(go.Scatter(x=t, y=gt_vx, mode='lines', name='True Vx', line=dict(color='black', dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=est_vx, mode='lines', name='Estimated Vx', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=gt_vy, mode='lines', name='True Vy', line=dict(color='black', dash='dot')), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=est_vy, mode='lines', name='Estimated Vy', line=dict(color='magenta')), row=2, col=1)

fig.update_layout(height=700, title_text="<b>Day 01: 2D Linear Kalman Filter Performance Evaluation</b>")
fig.show()
